In [11]:
import pandas as pd

# Sample data with duplicates
data = {
    'user_id': [1,2,2,3,3],
    'email': ['alice@test.com', 'bob@test.com', 'bob@test.com', 'charlie@test.com', 'charlie@test.com'],
    'signup_date': ['2026-01-01', '2026-01-02', '2026-01-05', '2026-01-03', '2026-01-01']
}
df = pd.DataFrame(data)

In [12]:
# 1. Identify / View duplicate rows
# 'first' (Default): Marks all True, first occurrence as False.
# 'last': Marks all True, leaving the very last occurrence as False.
# False: Marks absolutely every occurrence of the duplicated row as True
duplicate_mask = df.duplicated(subset=['user_id'], keep=False)
print("Duplicate rows:\n", df[duplicate_mask])


Duplicate rows:
    user_id             email signup_date
1        2      bob@test.com  2026-01-02
2        2      bob@test.com  2026-01-05
3        3  charlie@test.com  2026-01-03
4        3  charlie@test.com  2026-01-01


In [13]:
# 2. Drop duplicates, keeping the LATEST record based on a date column
df_sorted = df.sort_values('signup_date', ascending=False)
df_clean = df_sorted.drop_duplicates(subset=['user_id'], keep='first')


In [14]:
# 3. Drop exact duplicates across ALL columns

df_exact_clean = df.drop_duplicates()

# Another way
# df.drop_duplicates(subset=['col1', 'col2'], keep='first', inplace=False, ignore_index=False)
# Subset as to check duplicate by columns
# keep as First, Last, False
# inplace as True/False
# ignore_index is reset_index



Handle NaN while delete

In [15]:
import numpy as np

data = {
    'Product': ['Tablet', 'Tablet', 'Tablet'],
    'Price': [np.nan, 300, np.nan]
}
df = pd.DataFrame(data)

# Sorting pushes NaN to the bottom; keeps the valid $300 entry
df_clean_nan = df.sort_values(by='Price').drop_duplicates(subset=['Product'], keep='first')


Identify and Inspect Duplicates Before Dropping

In [17]:
data = {
    'Department': ['Sales', 'Sales', 'HR', 'HR', 'Sales'],
    'Employee': ['Amy', 'Ben', 'Charlie', 'David', 'Eva'],
    'Score': [85, 92, 78, 88, 95]
}
df = pd.DataFrame(data)


# Create a mask for all duplicate rows based on Department
duplicate_mask = df.duplicated(subset=['Department'], keep=False)

# View all rows that have a duplicate department to inspect them
all_duplicates = df.loc[duplicate_mask]
print(all_duplicates)

# Count how many duplicates exist per group
duplicate_counts = df[df.duplicated(subset=['Department'], keep='first')]['Department'].value_counts()
print(duplicate_counts)


  Department Employee  Score
0      Sales      Amy     85
1      Sales      Ben     92
2         HR  Charlie     78
3         HR    David     88
4      Sales      Eva     95
Department
Sales    2
HR       1
Name: count, dtype: int64


Transform or Aggregate Instead of Dropping

In [ ]:
data = {
    'Department': ['Sales', 'Sales', 'HR', 'HR', 'Sales'],
    'Employee': ['Amy', 'Amy', 'Charlie', 'David', 'Eva'],
    'Score': [85, 92, 78, 88, 95]
}
df = pd.DataFrame(data)

print(df)
print('----------------------------')
# Group by department and get the average score, preserving group rows as unique indices
df_avg_scores = df.groupby('Department')['Score'].mean().reset_index()
print(df_avg_scores)
print('----------------------------')
# Group by department and combine employee names into a comma-separated list
df_combined_names = df.groupby('Department')['Employee'].apply(', '.join).reset_index()
print(df_combined_names)

print('----------------------------')
# Aggregate both columns at once and reset the index
df_combined = df.groupby('Department').agg({
    'Score': 'mean',
    'Employee': lambda x: ', '.join(x.unique()),  # unique gives array of names, nunique gives counts
}).reset_index()

print(df_combined)

print('----------------------------')
df_combined2 = df.groupby("Department").agg(
        avg_sal=("Score", "mean"),
        max_sal=("Score", "max")).reset_index()
print(df_combined2)

print('----------------------------')
df2 = df_combined.merge(
            df.drop_duplicates(subset=['Department'], keep='first'),
            on='Department', suffixes=('_l', '_r')
        ).rename(columns={
                'Employee_l': 'Employees',
                'Score_l': 'Scores'
            }).drop(columns=['Employee_r', 'Score_r'])
print(df2)



  Department Employee  Score
0      Sales      Amy     85
1      Sales      Amy     92
2         HR  Charlie     78
3         HR    David     88
4      Sales      Eva     95
----------------------------
  Department      Score
0         HR  83.000000
1      Sales  90.666667
----------------------------
  Department        Employee
0         HR  Charlie, David
1      Sales   Amy, Amy, Eva
----------------------------
  Department      Score        Employee
0         HR  83.000000  Charlie, David
1      Sales  90.666667        Amy, Eva
----------------------------
  Department    avg_sal  max_sal
0         HR  83.000000       88
1      Sales  90.666667       95
----------------------------
  Department     Scores        Employee
0         HR  83.000000  Charlie, David
1      Sales  90.666667        Amy, Eva
